# Smart Gallery — ทดลองจากภาพจริงก่อนนำผลไปแสดงในแอป

Notebook นี้อธิบายและรัน **flow ที่แอปใช้อยู่จริง** ตามลำดับ ใช้ Python output ปกติ
ไม่มี dashboard, widgets หรือ server ให้เปิดเพิ่ม เลือกโฟลเดอร์แล้วดูรูปพร้อมผลได้เลย

**วิธี present:** อธิบายภาพรวม → ดูกระบวนการ 1–7 → รันส่วน 8 เลือกโฟลเดอร์ภาพ
→ ดูรูป/คะแนน/กลุ่ม/คำแนะนำ → เปิดแอปเพื่อแสดงการนำผลเดียวกันไปใช้งานจริง

**Run all:** เชื่อม Drive → ตรวจ path ในส่วน 2 → เมื่อถึงส่วน 8 เลือกหมายเลขโฟลเดอร์
หรือวาง path ภาพ ส่วนประเมิน Ground Truth และทดลองเปลี่ยนค่าปิดไว้จนกว่าจะเปิดเอง

ไฟล์ `best_parameters.json` ในโฟลเดอร์ `colab/` เป็นสำเนาค่าจากแอป ณ วันที่จัดทำ
ชื่อ algorithm ใช้ภาษาอังกฤษ ส่วนคำอธิบายและผลทดลองใช้ภาษาไทย


## 1. ภาพรวม flow ปัจจุบัน

```text
โฟลเดอร์ภาพ → อ่านภาพและเวลาถ่าย → แก้ orientation / RGB / resize ≤ 1024 px
            → Sharpness + Exposure + pHash features → Quality Score
            → แยกรูปมีเวลา / รูปไม่มีเวลา
                 มีเวลา: จัดกลุ่มด้วยเวลา → คงกลุ่มเดิม → รวมเพิ่มด้วย timed pHash
                 ไม่มีเวลา: จัดกลุ่มด้วย untimed pHash
            → Best Shot → near-tie review / quality gap → รูปและคำแนะนำ
```

- รูปมีเวลาที่ผ่านกรอบเวลาจัดกลุ่มได้เลย แม้ภาพขยับและ pHash ต่างกัน
- เมื่อเกินกรอบเวลา ใช้ `timed_phash_threshold` แยกจากเกณฑ์รูปไม่มีเวลา
- รวมกลุ่มเวลาเป็นหน่วยเดิมได้เมื่อ **ทุกคู่ข้ามกลุ่มผ่านเวลา หรือ timed pHash**
  ใช้ Complete-link เพื่อไม่ให้ภาพที่ผ่านแค่บางคู่ลากกลุ่มที่ไม่เข้าเกณฑ์เข้ามา
- รูปมีเวลากับไม่มีเวลาไม่รวมข้ามกัน
- ไม่ใช้ SSIM และไม่ใช้ Singleton Rescue / Group Merge แบบ notebook เก่าใน flow นี้
- คำแนะนำเป็น KEEP / REVIEW / RECOMMEND_DELETE โดยไม่มีการลบภาพต้นฉบับ


## 2. Setup และตำแหน่งไฟล์

วางโฟลเดอร์ `colab/` ในโปรเจกต์บน Drive เช่น

```text
MyDrive/Mini-Project-Image/
├── colab/presentation_results.ipynb
├── colab/best_parameters.json
├── dataset/                       # ภาพจริงเดิม ใช้ subfolder ใดก็ได้
│   └── metadata/labels.csv         # ใช้เมื่อเปิดส่วนประเมินเท่านั้น
└── module_outputs/app_flow/        # สร้างเมื่อเปิด export เท่านั้น
```

เปิด Notebook บน Colab แล้วรัน cell ติดตั้งครั้งแรก หากเก็บโปรเจกต์คนละที่
ให้แก้ `PROJECT_ROOT` ใน cell ตั้งค่าด้านล่าง ภาพ demo เลือกจากโฟลเดอร์อื่นได้


In [ ]:
# ติดตั้งสำหรับ Colab ไม่ต้องติดตั้ง FastAPI หรือรัน backend server
%pip -q install pillow pillow-heif opencv-python-headless numpy pandas ImageHash scikit-learn matplotlib


In [ ]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
from typing import Any
import hashlib
import json
import math
import os

import cv2 as cv
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import ExifTags, Image, ImageOps, UnidentifiedImageError
from pillow_heif import register_heif_opener
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from IPython.display import display

register_heif_opener()
MAX_SIDE = 1024
DARK_PIXEL_MAX = 10
HIGHLIGHT_PIXEL_MIN = 245
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.heic', '.heif'}


In [ ]:
# Colab จะถามอนุญาตเชื่อม Drive ใน cell นี้; local Jupyter ข้าม mount ให้เอง
IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass
if IN_COLAB:
    drive.mount('/content/drive')
else:
    print('Local runtime: ใช้ path ในเครื่องจาก cell ถัดไป')


In [ ]:
# แก้ path หลักจุดนี้ให้ตรงกับที่เก็บบน Drive
PROJECT_ROOT = Path(os.environ.get('SMART_GALLERY_PROJECT_ROOT', '/content/drive/MyDrive/Mini-Project-Image'))
COLAB_FOLDER = PROJECT_ROOT / 'colab'
PARAMS_PATH = COLAB_FOLDER / 'best_parameters.json'
DATASET_ROOT = PROJECT_ROOT / 'dataset'
LABELS_PATH = DATASET_ROOT / 'metadata' / 'labels.csv'
OUTPUT_DIR = PROJECT_ROOT / 'module_outputs' / 'app_flow'

# ว่างไว้ = เลือกจากรายการหรือวาง path ตอนถึงส่วน 8
IMAGE_FOLDER = os.environ.get('SMART_GALLERY_IMAGE_FOLDER', '')
MAX_PREVIEW_IMAGES = 12         # จำกัดเฉพาะจำนวนรูปที่แสดง; ยังวิเคราะห์ครบทั้งโฟลเดอร์
EXPORT_DEMO_RESULTS = False    # True เพื่อบันทึก CSV/JSON ใหม่ใน module_outputs/app_flow

# การประเมินเป็นขั้นเสริม ไม่จำเป็นสำหรับ demo ภาพทั่วไป
RUN_GROUND_TRUTH_EVALUATION = False
EVALUATION_SPLIT = 'development'  # development / final_test / all
RUN_PARAMETER_COMPARISON = False

print('Project:', PROJECT_ROOT)
print('Parameters:', PARAMS_PATH)
print('Ground Truth (optional):', LABELS_PATH)


### 2.1 โหลด parameter เดียวกับแอป

โหลด JSON โดยตรง ไม่บังคับค่า 29/30/Average แบบ notebook เดิม และไม่เขียนทับไฟล์
ค่า `development_*` และ `selection` ใน JSON เป็นประวัติการทดลองเก่า
ไม่ใช่คะแนนความแม่นยำของ flow ใหม่นี้


In [ ]:
def validate_parameters(params: dict) -> dict:
    sharp = params["sharpness"]
    exposure = params["exposure"]
    similarity = params["similarity"]
    quality = params["quality_score"]
    gap = params["recommendation"]["duplicate_quality_gap"]

    def number(value, low, high, name):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError(f"{name} must be a number")
        if not math.isfinite(value) or not low <= value <= high:
            raise ValueError(f"{name} must be between {low} and {high}")

    if sharp["method"] not in {"laplacian_score", "tenengrad_score"}:
        raise ValueError("Unknown sharpness method")
    number(sharp["threshold"], 0, float("inf"), "Sharpness threshold")
    number(exposure["low_brightness"], 0, 255, "Low brightness")
    number(exposure["high_brightness"], 0, 255, "High brightness")
    if exposure["low_brightness"] >= exposure["high_brightness"]:
        raise ValueError("Low brightness must be below high brightness")
    for key in ("dark_clip_threshold", "highlight_clip_threshold"):
        number(exposure[key], 1e-9, 1, key)
    if similarity["strategy"] not in {"exif_phash", "exif_time_or_phash"}:
        raise ValueError("Similarity strategy must be exif_phash or exif_time_or_phash")
    if similarity.get("missing_exif_policy", "strict") not in {"strict", "phash"}:
        raise ValueError("Missing EXIF policy must be strict or phash")
    if similarity.get("linkage", "complete") not in {"complete", "average"}:
        raise ValueError("Similarity linkage must be complete or average")
    if similarity.get("ssim_threshold") is not None:
        raise ValueError(
            "Legacy SSIM threshold detected; use the latest EXIF+pHash parameters"
        )
    number(similarity["phash_threshold"], 0, 64, "pHash threshold")
    if int(similarity["phash_threshold"]) != similarity["phash_threshold"]:
        raise ValueError("pHash threshold must be an integer")
    timed_phash_threshold = similarity.get("timed_phash_threshold")
    if timed_phash_threshold is not None:
        number(timed_phash_threshold, 0, 64, "Timed pHash threshold")
        if int(timed_phash_threshold) != timed_phash_threshold:
            raise ValueError("Timed pHash threshold must be an integer")
    # Time-only grouping and the legacy refinements need a finite base time window.
    number(similarity.get("exif_time_window_seconds"), 0, float("inf"), "EXIF window")
    for key in ("sharpness_weight", "exposure_weight"):
        number(quality[key], 0, 1, key)
    if not math.isclose(
        sum(quality[k] for k in ("sharpness_weight", "exposure_weight")),
        1.0,
        abs_tol=1e-6,
    ):
        raise ValueError("Quality weights must sum to 1")
    number(gap, 0, 100, "Quality gap")
    number(
        params["recommendation"].get("best_shot_tie_margin", 2.0),
        0,
        float("inf"),
        "Best Shot tie margin",
    )
    return params


In [ ]:
def load_demo_parameters(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'ไม่พบ {path} — วาง best_parameters.json จากโฟลเดอร์ colab แล้วตรวจ PROJECT_ROOT')
    selected = json.loads(path.read_text(encoding='utf-8'))
    selected['similarity'].setdefault('linkage', 'complete')
    selected['similarity'].setdefault('missing_exif_policy', 'strict')
    selected['recommendation'].setdefault('best_shot_tie_margin', 2.0)
    validate_parameters(selected)
    if selected['similarity']['strategy'] != 'exif_time_or_phash':
        raise ValueError('Notebook ฉบับนี้แสดง flow แอปปัจจุบัน: ใช้ strategy exif_time_or_phash')
    return selected

PARAMS = load_demo_parameters(PARAMS_PATH)
PARAMETER_SHA256 = hashlib.sha256(PARAMS_PATH.read_bytes()).hexdigest()
runtime_settings = [
    ('Sharpness method', PARAMS['sharpness']['method']),
    ('Sharpness threshold', PARAMS['sharpness']['threshold']),
    ('Brightness low / high', f"{PARAMS['exposure']['low_brightness']} / {PARAMS['exposure']['high_brightness']}"),
    ('Dark / highlight clipping', f"{PARAMS['exposure']['dark_clip_threshold']} / {PARAMS['exposure']['highlight_clip_threshold']}"),
    ('EXIF window (seconds)', PARAMS['similarity']['exif_time_window_seconds']),
    ('Untimed pHash threshold', PARAMS['similarity']['phash_threshold']),
    ('Timed pHash threshold', PARAMS['similarity'].get('timed_phash_threshold')),
    ('Untimed linkage', PARAMS['similarity']['linkage']),
    ('Sharpness / exposure weights', f"{PARAMS['quality_score']['sharpness_weight']} / {PARAMS['quality_score']['exposure_weight']}"),
    ('Best Shot tie margin', PARAMS['recommendation']['best_shot_tie_margin']),
    ('Duplicate quality gap', PARAMS['recommendation']['duplicate_quality_gap']),
]
display(pd.DataFrame(runtime_settings, columns=['Parameter', 'Value']))


### 2.2 แผนภาพสำหรับอธิบายก่อนดูโค้ด

ส่วนนี้แสดง diagram ธรรมดาใน output ของ Notebook ไม่มี UI ให้ควบคุม


In [ ]:
def show_system_flow():
    fig, ax = plt.subplots(figsize=(14, 7.8))
    ax.set(xlim=(0, 14), ylim=(0, 7.8))
    ax.axis('off')
    boxes = {
        'input': (1.5, 7.0, 'Image folder'),
        'load': (5.0, 7.0, 'Load / EXIF orientation\nRGB / resize <= 1024'),
        'quality': (9.4, 7.0, 'Sharpness + Exposure\nQuality score'),
        'split': (6.8, 5.5, 'Split by capture timestamp'),
        'timed': (3.1, 4.0, 'Timed photos\nComplete-link by time'),
        'merge': (3.1, 2.4, 'Keep time groups\n+ separate timed pHash'),
        'untimed': (10.2, 4.0, 'Untimed photos\npHash clustering'),
        'best': (6.9, 1.0, 'Best Shot\nNear-tie / quality gap'),
        'output': (11.5, 1.0, 'Images + scores\nShort reasons'),
    }
    for key, (x, y, label) in boxes.items():
        ax.text(x, y, label, ha='center', va='center', fontsize=10,
                bbox={'boxstyle': 'round,pad=0.55', 'facecolor': '#eff6ff' if key not in {'best', 'output'} else '#ecfdf5', 'edgecolor': '#94a3b8'})
    arrows = [
        ((2.65, 7.0), (3.7, 7.0)), ((6.35, 7.0), (8.0, 7.0)),
        ((9.4, 6.5), (7.8, 5.9)),
        ((5.6, 5.2), (3.9, 4.5)), ((8.0, 5.2), (9.4, 4.5)),
        ((3.1, 3.5), (3.1, 2.9)), ((4.7, 2.15), (5.65, 1.4)),
        ((10.2, 3.5), (8.1, 1.45)), ((8.4, 1.0), (10.2, 1.0)),
    ]
    for start, end in arrows:
        ax.annotate('', xy=end, xytext=start,
                    arrowprops={'arrowstyle': '->', 'color': '#475569', 'lw': 1.4})
    ax.text(7.0, 0.15, 'Timed and untimed pools never mix. Original time groups are not split.', ha='center', fontsize=10, color='#475569')
    plt.tight_layout()
    display(fig)
    plt.close(fig)

show_system_flow()


## 3. อ่านภาพและเตรียมข้อมูล

เรียงขั้นเหมือนแอป: อ่านเวลาถ่ายจาก EXIF รวม nested IFD ของไฟล์มือถือ
→ แก้ EXIF orientation → RGB → resize โดยคงอัตราส่วน ด้านยาวสุด 1024 px
ไม่แก้ไฟล์ต้นฉบับ รูปที่ไม่มีเวลาถ่ายเก็บ `None` เพื่อเข้าชุด untimed ภายหลัง


In [ ]:
def parse_exif_datetime(value: Any) -> datetime | None:
    if not value:
        return None

    for fmt in ("%Y:%m:%d %H:%M:%S", "%Y-%m-%d %H:%M:%S"):
        try:
            return datetime.strptime(str(value), fmt)
        except ValueError:
            pass

    return None


def load_image(path: Path, max_side: int = MAX_SIDE) -> dict[str, Any]:
    with Image.open(path) as pil:
        exif_raw = pil.getexif()
        exif = (
            {ExifTags.TAGS.get(k, str(k)): v for k, v in exif_raw.items()}
            if exif_raw
            else {}
        )
        if exif_raw:
            # Phone/camera DateTimeOriginal commonly lives in the Exif sub-IFD.
            capture_exif = exif_raw.get_ifd(ExifTags.IFD.Exif)
            exif.update(
                {
                    ExifTags.TAGS.get(key, str(key)): value
                    for key, value in capture_exif.items()
                }
            )

        pil = ImageOps.exif_transpose(pil).convert("RGB")
        original_width, original_height = pil.size
        rgb = np.array(pil)

    if max(rgb.shape[:2]) > max_side:
        scale = max_side / max(rgb.shape[:2])
        new_size = (
            max(1, round(rgb.shape[1] * scale)),
            max(1, round(rgb.shape[0] * scale)),
        )
        rgb = cv.resize(rgb, new_size, interpolation=cv.INTER_AREA)

    taken_at = parse_exif_datetime(
        exif.get("DateTimeOriginal")
        or exif.get("DateTimeDigitized")
        or exif.get("DateTime")
    )

    return {
        "path": str(path),
        "filename": path.name,
        "rgb": rgb,
        "file_size_bytes": path.stat().st_size,
        "taken_at": taken_at,
        "original_width": original_width,
        "original_height": original_height,
    }


def to_gray(rgb: np.ndarray) -> np.ndarray:
    return cv.cvtColor(rgb, cv.COLOR_RGB2GRAY)


## 4. Feature extraction

- **Sharpness:** Variance of Laplacian เป็นวิธีหลัก; Tenengrad เก็บเป็นตัวเลือกเปรียบเทียบ
- **Exposure:** mean/median ของ Y luminance, สัดส่วน pixel มืด ≤ 10 และสว่าง ≥ 245
- **pHash:** grayscale + DCT สร้าง 64-bit hash; Hamming distance น้อยหมายถึงแฮชใกล้กัน
- ระบุภาพด้วย path จึงรองรับชื่อไฟล์ซ้ำจากคนละ subfolder


In [ ]:
def sharpness_features(rgb: np.ndarray) -> dict[str, float]:
    gray = to_gray(rgb).astype(np.float64)

    lap = cv.Laplacian(gray, cv.CV_64F)
    gx = cv.Sobel(gray, cv.CV_64F, 1, 0, ksize=3)
    gy = cv.Sobel(gray, cv.CV_64F, 0, 1, ksize=3)

    return {
        "laplacian_score": float(lap.var()),
        "tenengrad_score": float(np.mean(gx**2 + gy**2)),
    }


def exposure_features(rgb: np.ndarray) -> dict[str, float]:
    y = cv.cvtColor(rgb, cv.COLOR_RGB2YCrCb)[:, :, 0]

    return {
        "brightness_mean": float(np.mean(y)),
        "brightness_median": float(np.median(y)),
        "dark_clipping_ratio": float(np.mean(y <= DARK_PIXEL_MAX)),
        "highlight_clipping_ratio": float(np.mean(y >= HIGHLIGHT_PIXEL_MIN)),
    }


def phash_value(rgb: np.ndarray):
    return imagehash.phash(Image.fromarray(rgb), hash_size=8)


def extract_features(path: Path) -> dict[str, Any]:
    item = load_image(path)
    sharp = sharpness_features(item["rgb"])
    exposure = exposure_features(item["rgb"])

    return {
        "filename": item["filename"],
        "path": item["path"],
        "file_size_bytes": item["file_size_bytes"],
        "taken_at": (item["taken_at"].isoformat() if item["taken_at"] else None),
        "original_width": item["original_width"],
        "original_height": item["original_height"],
        **sharp,
        **exposure,
        "phash": str(phash_value(item["rgb"])),
    }


## 5. Classification และ Quality Score

### 5.1 Sharpness / Exposure labels

Sharp เมื่อ raw sharpness ≥ threshold ส่วนแสงตรวจ underexposed ก่อน overexposed
ตามโค้ดแอป ค่าเฉลี่ยปกติยังอาจถูกจัดว่าสว่างเกิน หาก highlight clipping ถึงเกณฑ์

Dataset ความคมชัดมี 4 ประเภท: Sharp / Slight Blur / Motion Blur / Defocus Blur
แต่ฟังก์ชันนี้ทำนายเพียง `sharp` / `blur` โดยรวมเบลอทั้ง 3 ประเภทเป็น `blur`
ส่วน 9 จะแจกแจงผลแยกตามประเภทเดิมครบทุกภาพ ไม่ใช่การทำนายชนิดเบลอ 4 class
Exposure ทำนาย 3 class และ Similarity ใช้ Group ID เพื่อประเมินการจัดกลุ่ม


In [ ]:
def classify_sharpness(
    row: pd.Series,
    params: dict[str, Any],
) -> str:
    cfg = params["sharpness"]

    return "sharp" if float(row[cfg["method"]]) >= float(cfg["threshold"]) else "blur"


def classify_exposure(
    row: pd.Series,
    params: dict[str, Any],
) -> str:
    cfg = params["exposure"]

    if (
        row["brightness_mean"] <= cfg["low_brightness"]
        or row["dark_clipping_ratio"] >= cfg["dark_clip_threshold"]
    ):
        return "underexposed"

    if (
        row["brightness_mean"] >= cfg["high_brightness"]
        or row["highlight_clipping_ratio"] >= cfg["highlight_clip_threshold"]
    ):
        return "overexposed"

    return "normal"


### 5.2 Normalize คะแนน (0–100)

\[
S = 100 \frac{\max(0, raw)}{\max(0, raw)+threshold}
\]

Exposure score มาจากช่วง brightness และ clipping penalty ตามฟังก์ชันด้านล่าง
**คะแนน Exposure 100 ไม่ได้แปลว่า label ต้อง normal เสมอไป** เพราะขอบเงื่อนไข
classification และ scoring ต่างกัน เวลาแสดง demo จึงใช้สถานะแสงแทนคะแนนแสง


In [ ]:
def sharpness_score_0_100(
    row: pd.Series,
    params: dict[str, Any],
) -> float:
    cfg = params["sharpness"]

    raw = max(0.0, float(row[cfg["method"]]))
    threshold = max(1e-9, float(cfg["threshold"]))

    return float(
        np.clip(
            100.0 * raw / (raw + threshold),
            0,
            100,
        )
    )


def exposure_score_0_100(
    row: pd.Series,
    params: dict[str, Any],
) -> float:
    cfg = params["exposure"]

    mean = float(row["brightness_mean"])
    low = float(cfg["low_brightness"])
    high = float(cfg["high_brightness"])

    if mean < low:
        brightness_score = 100.0 * max(0.0, mean / max(low, 1e-9))
    elif mean > high:
        brightness_score = 100.0 * max(
            0.0,
            (255.0 - mean) / max(255.0 - high, 1e-9),
        )
    else:
        brightness_score = 100.0

    dark_ratio = float(row["dark_clipping_ratio"])
    highlight_ratio = float(row["highlight_clipping_ratio"])

    dark_thr = max(
        float(cfg["dark_clip_threshold"]),
        1e-9,
    )
    high_thr = max(
        float(cfg["highlight_clip_threshold"]),
        1e-9,
    )

    clip_penalty = 25.0 * max(
        max(0.0, dark_ratio / dark_thr - 1.0),
        max(
            0.0,
            highlight_ratio / high_thr - 1.0,
        ),
    )

    return float(
        np.clip(
            brightness_score - clip_penalty,
            0,
            100,
        )
    )


### 5.3 รวมคะแนนคุณภาพ

\[
Q = \frac{w_s S + w_e E}{w_s+w_e}
\]

น้ำหนักอ่านจาก JSON ไม่ใช้ตัวเลขฝังในคำอธิบายเป็นตัวตัดสิน คะแนนนี้เป็น heuristic
สำหรับจัดอันดับภายในกลุ่ม ไม่ใช่เปอร์เซ็นต์ความแม่นยำของโมเดล


In [ ]:
def add_quality_scores(
    df: pd.DataFrame,
    params: dict[str, Any],
) -> pd.DataFrame:
    result = df.copy()

    result["sharpness_label"] = result.apply(
        lambda row: classify_sharpness(row, params),
        axis=1,
    )
    result["exposure_label_pred"] = result.apply(
        lambda row: classify_exposure(row, params),
        axis=1,
    )
    result["sharpnessScore"] = result.apply(
        lambda row: sharpness_score_0_100(row, params),
        axis=1,
    )
    result["exposureScore"] = result.apply(
        lambda row: exposure_score_0_100(row, params),
        axis=1,
    )

    sw = float(
        params["quality_score"].get(
            "sharpness_weight",
            0.60,
        )
    )
    ew = float(
        params["quality_score"].get(
            "exposure_weight",
            0.40,
        )
    )

    total = sw + ew

    result["qualityScore"] = (
        sw * result["sharpnessScore"] + ew * result["exposureScore"]
    ) / total

    return result


## 6. Similarity grouping — แยกสองชุดภาพ

### 6.1 รูปมีเวลา: จัดตามเวลาก่อน แล้วพิจารณา timed pHash

เริ่มด้วย Complete-link ของระยะเวลา ไม่เกิน `exif_time_window_seconds` ทุกคู่ในกลุ่มตั้งต้น
คู่ที่ผ่านเวลาไม่ต้องผ่าน pHash ส่วนคู่ที่เกินเวลาใช้ `timed_phash_threshold` แยกต่างหาก

เก็บกลุ่มเวลาเดิมเป็นหน่วยเดียว รวมเพิ่มได้เมื่อทุกคู่ข้ามกลุ่มผ่าน **เวลา OR timed pHash**
ใช้ Complete-link ป้องกันการรวมเป็นลูกโซ่จากคู่ที่เข้าเกณฑ์เพียงบางคู่
จึงมีโอกาสรวมภาพคล้ายที่ถ่ายห่างกัน โดยไม่ทำให้กลุ่มเวลาที่ได้แล้วแยกออก
หากเกณฑ์ timed pHash เป็น `null` หรือไม่มี key จะใช้เวลาอย่างเดียว และไม่มีเพดานเวลาใหม่
สำหรับคู่ที่ตรวจ timed pHash


In [ ]:
def _merge_timed_phash_groups(seed_groups, pair_distances, threshold):
    """Merge whole time groups only when every cross-group pair is eligible."""
    distances = np.zeros((len(seed_groups), len(seed_groups)), dtype=float)
    for a, b in combinations(range(len(seed_groups)), 2):
        distance = max(
            pair_distances[tuple(sorted((i, j)))]
            for i in seed_groups[a]
            for j in seed_groups[b]
        )
        distances[a, b] = distances[b, a] = distance
    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="complete",
        distance_threshold=float(threshold) + 1e-09,
    ).fit_predict(distances)
    members_by_cluster = {}
    seeds_by_cluster = {}
    for members, label in zip(seed_groups, labels):
        members_by_cluster.setdefault(label, []).extend(members)
        seeds_by_cluster[label] = seeds_by_cluster.get(label, 0) + 1
    merged_indices = {
        index
        for label, members in members_by_cluster.items()
        if seeds_by_cluster[label] > 1
        for index in members
    }
    return list(members_by_cluster.values()), merged_indices


### 6.2 รูปไม่มีเวลา: ใช้ untimed pHash แยกกัน

ใช้ `phash_threshold` และ `linkage` จาก JSON เฉพาะรูปไม่มีเวลา
ถ้า missing policy เป็น `strict` รูปเหล่านี้จะไม่เข้ากลุ่ม แต่ยังได้คะแนนคุณภาพ
ทั้งสองชุดไม่รวมกัน แม้ภาพจะเหมือนกัน

ตาราง pairs เป็นผลตรวจแต่ละคู่ ส่วนกลุ่มสุดท้ายต้องผ่านเงื่อนไขการรวมกลุ่มด้วย
`is_similar=True` เพียงคู่เดียวจึงไม่ได้รับประกันว่าทั้งสองกลุ่มจะถูกรวม


In [ ]:
def _build_time_or_phash_groups(df, params, missing_exif_policy):
    """Keep time groups intact, optionally bridge them using a separate pHash rule."""
    cfg = params["similarity"]
    result = df.copy().reset_index(drop=True)
    records = result.to_dict("records")
    timestamps = [pd.to_datetime(r.get("taken_at"), errors="coerce") for r in records]
    timed = [i for i, timestamp in enumerate(timestamps) if pd.notna(timestamp)]
    untimed = [i for i, timestamp in enumerate(timestamps) if pd.isna(timestamp)]
    pools = [(timed, "exif_time", float(cfg["exif_time_window_seconds"]), "complete")]
    if missing_exif_policy == "phash":
        pools.append(
            (
                untimed,
                "phash",
                float(cfg["phash_threshold"]),
                cfg.get("linkage", "complete"),
            )
        )

    clusters = []
    pair_rows = []
    timed_phash_threshold = cfg.get("timed_phash_threshold")
    timed_pair_distances = {}
    timed_hashes = {}
    result["timedPhashMatch"] = False
    for indices, strategy, threshold, linkage in pools:
        if len(indices) < 2:
            continue
        distances = np.zeros((len(indices), len(indices)), dtype=float)
        hashes = (
            [imagehash.hex_to_hash(str(records[i]["phash"])) for i in indices]
            if strategy == "phash"
            else None
        )
        for local_i, local_j in combinations(range(len(indices)), 2):
            i, j = indices[local_i], indices[local_j]
            if strategy == "exif_time":
                time_diff = abs((timestamps[i] - timestamps[j]).total_seconds())
                hamming = None
                distance = time_diff
            else:
                time_diff = None
                hamming = int(hashes[local_i] - hashes[local_j])
                distance = float(hamming)
            pair_pass = bool(distance <= threshold)
            pair_strategy = strategy
            if strategy == "exif_time" and timed_phash_threshold is not None:
                if pair_pass:
                    # Time-matched photos never have to pass pHash.
                    timed_pair_distances[i, j] = 0.0
                else:
                    for index in (i, j):
                        if index not in timed_hashes:
                            timed_hashes[index] = imagehash.hex_to_hash(
                                str(records[index]["phash"])
                            )
                    hamming = int(timed_hashes[i] - timed_hashes[j])
                    timed_pair_distances[i, j] = float(hamming)
                    pair_pass = bool(hamming <= timed_phash_threshold)
                    pair_strategy = "timed_phash"
            distances[local_i, local_j] = distances[local_j, local_i] = distance
            pair_rows.append(
                {
                    "image_a": records[i]["filename"],
                    "image_b": records[j]["filename"],
                    "strategy": pair_strategy,
                    "linkage": linkage,
                    "time_difference_seconds": time_diff,
                    "hamming_distance": hamming,
                    "is_similar": pair_pass,
                }
            )
        labels = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage=linkage,
            distance_threshold=threshold + 1e-09,
        ).fit_predict(distances)
        members_by_cluster = {}
        for index, label in zip(indices, labels):
            members_by_cluster.setdefault(label, []).append(index)
        pool_groups = list(members_by_cluster.values())
        if (
            strategy == "exif_time"
            and timed_phash_threshold is not None
            and len(pool_groups) > 1
        ):
            pool_groups, merged_indices = _merge_timed_phash_groups(
                pool_groups, timed_pair_distances, timed_phash_threshold
            )
            result.loc[sorted(merged_indices), "timedPhashMatch"] = True
        clusters.extend(members for members in pool_groups if len(members) >= 2)

    group_by_index = {}
    for number, members in enumerate(sorted(clusters, key=min), start=1):
        for index in members:
            group_by_index[index] = f"G{number:03d}"
    result["similarityGroup"] = [group_by_index.get(i) for i in range(len(result))]
    return result, pd.DataFrame(pair_rows)


## 7. Best Shot และ Recommendation

1. ภายในแต่ละกลุ่มเลือกคะแนนสูงสุดเป็น Best Shot เบื้องต้น; รูปเดี่ยวไม่มี badge นี้
2. ถ้าอันดับ 1 กับ 2 ห่าง **น้อยกว่า** tie margin ให้สมาชิกที่ใกล้คะแนนสูงสุด
   ภายใน margin เป็น REVIEW (`bestShotNeedsReview=True`) ไม่ใช่ REVIEW ทุกสมาชิกเสมอไป
3. สมาชิกที่เหลือซึ่งต่ำกว่าคะแนนสูงสุด **ตั้งแต่** quality gap ขึ้นไปเป็น RECOMMEND_DELETE
   สมาชิกที่ใกล้กว่านั้นเป็น REVIEW; Best Shot ที่ไม่ near-tie เป็น KEEP
4. รูปเดี่ยวเป็น KEEP เมื่อ sharp และ normal exposure นอกนั้น REVIEW

คำแนะนำและเหตุผลใช้ฟังก์ชันเดียวกับแอป ไม่มีการลบไฟล์


In [ ]:
def add_best_shot_and_recommendation(df, params):
    result = df.copy()
    result["bestShot"] = False
    result["bestShotNeedsReview"] = False
    tie_margin = float(params["recommendation"].get("best_shot_tie_margin", 2.0))
    delete_gap = float(params["recommendation"].get("duplicate_quality_gap", 5.0))
    if tie_margin < 0:
        raise ValueError("best_shot_tie_margin cannot be negative")
    grouped = result[result["similarityGroup"].notna()].groupby("similarityGroup")
    for _, group in grouped:
        ranked = group["qualityScore"].sort_values(ascending=False)
        best_idx = ranked.index[0]
        result.loc[best_idx, "bestShot"] = True
        if len(ranked) >= 2:
            score_gap = float(ranked.iloc[0] - ranked.iloc[1])
            if score_gap < tie_margin:
                near_tie_idxs = ranked.index[ranked.iloc[0] - ranked < tie_margin]
                result.loc[near_tie_idxs, "bestShotNeedsReview"] = True
    recommendations = []
    reasons_all = []
    for _, row in result.iterrows():
        reasons = []
        group_id = row["similarityGroup"]
        time_group = params["similarity"][
            "strategy"
        ] == "exif_time_or_phash" and pd.notna(
            pd.to_datetime(row.get("taken_at"), errors="coerce")
        )
        timed_phash_group = time_group and bool(row.get("timedPhashMatch", False))
        if row["sharpness_label"] == "blur":
            reasons.append("Possible blur detected")
        else:
            reasons.append("Sharpness is acceptable")
        if row["exposure_label_pred"] != "normal":
            reasons.append(f"Possible {row['exposure_label_pred']}")
        else:
            reasons.append("Exposure is within the normal range")
        if time_group:
            reasons.append(
                "Capture-time groups combined using the timed pHash threshold."
                if timed_phash_group
                else "Capture-time grouping only; pHash was not used."
            )
        if pd.notna(group_id):
            group = result[result["similarityGroup"] == group_id]
            best_quality = float(group["qualityScore"].max())
            if bool(row["bestShotNeedsReview"]):
                recommendations.append("REVIEW")
                if bool(row["bestShot"]):
                    reasons.append(
                        "Provisional best shot; near-tied quality — review composition"
                    )
                else:
                    reasons.append(
                        "Best-shot scores are too close to recommend deletion"
                    )
            elif bool(row["bestShot"]):
                recommendations.append("KEEP")
                reasons.append(
                    "Best shot in a timed-photo pHash group"
                    if timed_phash_group
                    else (
                        "Best shot in a capture-time group"
                        if time_group
                        else "Best shot in a similar-photo group"
                    )
                )
            elif best_quality - float(row["qualityScore"]) >= delete_gap:
                recommendations.append("RECOMMEND_DELETE")
                reasons.append(
                    "Timed-photo pHash group contains a higher quality photo"
                    if timed_phash_group
                    else (
                        "Photo in the same capture-time group has a higher quality score"
                        if time_group
                        else "Similar photo exists with a higher quality score"
                    )
                )
            else:
                recommendations.append("REVIEW")
                reasons.append(
                    "Timed-photo pHash group detected, but quality is close to the best shot"
                    if timed_phash_group
                    else (
                        "Capture-time group detected, but quality is close to the best shot"
                        if time_group
                        else "Similar photo detected, but quality is close to the best shot"
                    )
                )
        elif (
            row["sharpness_label"] == "sharp" and row["exposure_label_pred"] == "normal"
        ):
            recommendations.append("KEEP")
            reasons.append(
                "No other photo grouped within the capture-time window"
                if time_group
                else "No similar photo detected"
            )
        else:
            recommendations.append("REVIEW")
            reasons.append("Unique image — manual review recommended")
        reasons_all.append(reasons)
    result["recommendation"] = recommendations
    result["reasons"] = reasons_all
    return result


### 7.1 ต่อ flow ให้รันได้ตั้งแต่ features จนจบ

ตัว orchestration ด้านล่างแสดงเฉพาะ branch `exif_time_or_phash` ที่แอปใช้จริง
จึงไม่ใส่ branch เปรียบเทียบแบบเก่าและ Singleton Rescue ลงในส่วน present
ฟังก์ชันคำนวณก่อนหน้านี้คัดจาก source ของแอปโดยตรง และมี regression test เทียบผล


In [ ]:
def build_similarity_groups(features, params, missing_exif_policy=None):
    validate_parameters(params)
    if params['similarity']['strategy'] != 'exif_time_or_phash':
        raise ValueError('ส่วน present นี้ใช้เฉพาะ flow แอป exif_time_or_phash')
    policy = missing_exif_policy if missing_exif_policy is not None else params['similarity'].get('missing_exif_policy', 'strict')
    if policy not in {'strict', 'phash'}:
        raise ValueError('missing_exif_policy ต้องเป็น strict หรือ phash')
    return _build_time_or_phash_groups(features, params, policy)

def run_core_pipeline(features, params):
    scored = add_quality_scores(features, params)
    grouped, pairs = build_similarity_groups(scored, params)
    results = add_best_shot_and_recommendation(grouped, params)
    return results, pairs

def analyze_features(features, params):
    if features.empty:
        return features.copy(), pd.DataFrame()
    return run_core_pipeline(features, params)


## 8. ทดลองกับโฟลเดอร์ภาพจริง — output ธรรมดา

เลือกจากรายการหรือวาง path โฟลเดอร์จาก sidebar ของ Colab (Mount Drive แล้ว Copy path)
ระบบค้นภาพใน subfolder ด้วย และข้ามไฟล์ภาพที่อ่านไม่ได้พร้อมแจ้งชื่อ
Demo ใช้ขีดจำกัดแบบแอป: ≤ 200 ภาพ, ≤ 30 MiB/ไฟล์ และ ≤ 200 MiB รวม
การประเมิน dataset ในส่วน 9 ไม่ใช้ขีดจำกัด batch ของหน้าอัปโหลดนี้

`MAX_PREVIEW_IMAGES` จำกัดเฉพาะรูปที่แสดงเพื่อให้ present กระชับ ผลคำนวณยังครบทุกภาพ
เลือกแสดงแค่กลุ่มใดกลุ่มหนึ่งได้โดยส่ง `group_id` ให้ `show_photo_results()`


In [ ]:
def list_image_files(folder):
    folder = Path(folder).expanduser().resolve()
    if not folder.is_dir():
        raise FileNotFoundError(f'ไม่พบโฟลเดอร์ภาพ: {folder}')
    return sorted((path for path in folder.rglob('*') if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS), key=lambda path: str(path).casefold())

def choose_image_folder(folder_value='', dataset_root=None):
    if str(folder_value).strip():
        selected = Path(str(folder_value).strip()).expanduser()
    else:
        root = Path(dataset_root) if dataset_root is not None else DATASET_ROOT
        candidates = []
        if root.is_dir():
            candidates = [root] + sorted(path for path in root.iterdir() if path.is_dir() and path.name != 'metadata')
        print('เลือกโฟลเดอร์ภาพ (รวม subfolder):')
        for index, candidate in enumerate(candidates, start=1):
            print(f'  {index}. {candidate}')
        answer = input('พิมพ์หมายเลข หรือวาง path โฟลเดอร์: ').strip().strip('"').strip("'")
        if answer.isdigit() and 1 <= int(answer) <= len(candidates):
            selected = candidates[int(answer) - 1]
        elif answer:
            selected = Path(answer).expanduser()
        else:
            raise ValueError('ยังไม่ได้เลือกโฟลเดอร์: ใส่ IMAGE_FOLDER หรือเลือกหมายเลขแล้วรัน cell อีกครั้ง')
    if not selected.is_dir():
        raise FileNotFoundError(f'ไม่พบโฟลเดอร์: {selected}')
    return selected.resolve()

def extract_features_batch(paths, relative_to=None, enforce_demo_limits=False):
    paths = [Path(path) for path in paths]
    if enforce_demo_limits:
        if not 1 <= len(paths) <= 200:
            raise ValueError(f'Demo รองรับ 1–200 ภาพ พบ {len(paths)} ภาพ เลือก subfolder ที่เล็กลง')
        sizes = [path.stat().st_size for path in paths]
        if any(size > 30 * 1024 * 1024 for size in sizes):
            raise ValueError('มีไฟล์ใหญ่กว่า 30 MiB ซึ่งเกินขีดจำกัด demo')
        if sum(sizes) > 200 * 1024 * 1024:
            raise ValueError('ขนาดรวมเกิน 200 MiB ซึ่งเกินขีดจำกัด demo')
    features, issues = [], []
    for path in paths:
        try:
            row = extract_features(path)
            row['relative_path'] = path.resolve().relative_to(Path(relative_to).resolve()).as_posix() if relative_to is not None else path.name
            features.append(row)
        except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError) as exc:
            issues.append({'filename': path.name, 'path': str(path), 'error': str(exc)})
    return pd.DataFrame(features), pd.DataFrame(issues)

def analyze_folder(folder, params):
    folder = Path(folder).resolve()
    features, issues = extract_features_batch(list_image_files(folder), relative_to=folder, enforce_demo_limits=True)
    if features.empty:
        raise ValueError(f'ไม่มีภาพที่อ่านได้ ข้าม {len(issues)} ไฟล์ ตรวจชนิดไฟล์และ path')
    results, pairs = analyze_features(features, deepcopy(params))
    return features, results, pairs, issues


In [ ]:
EXPOSURE_TEXT = {'underexposed': 'มืดเกินไป', 'normal': 'ปกติ', 'overexposed': 'สว่างเกินไป'}
RECOMMENDATION_TEXT = {'KEEP': 'แนะนำให้เก็บ', 'REVIEW': 'ควรตรวจสอบ', 'RECOMMEND_DELETE': 'แนะนำให้ลบ'}

def short_reason(row, results):
    if bool(row['bestShotNeedsReview']):
        return 'คะแนนใกล้รูปที่ดีที่สุด ควรตรวจสอบก่อนตัดสินใจ'
    if bool(row['bestShot']):
        return 'คะแนนสูงสุดในกลุ่ม'
    if row['recommendation'] == 'RECOMMEND_DELETE':
        best = float(results.loc[results['similarityGroup'].eq(row['similarityGroup']), 'qualityScore'].max())
        return f'มีรูปในกลุ่มที่ได้คะแนนสูงกว่า {best - float(row["qualityScore"]):.2f} คะแนน'
    if pd.notna(row['similarityGroup']):
        return 'อยู่ในกลุ่มเดียวกัน แต่คุณภาพใกล้เคียงกัน'
    return 'รูปเดี่ยว คุณภาพอยู่ในเกณฑ์' if row['recommendation'] == 'KEEP' else 'รูปเดี่ยว ควรตรวจคุณภาพด้วยตนเอง'

def summary_table(results):
    counts = results['recommendation'].value_counts()
    saved = results.loc[results['recommendation'].eq('RECOMMEND_DELETE'), 'file_size_bytes'].sum() / (1024 * 1024)
    return pd.DataFrame([
        ('วิเคราะห์แล้ว', len(results)), ('จำนวนกลุ่ม', results['similarityGroup'].nunique()),
        ('ไม่มีเวลาถ่าย', int(results['taken_at'].isna().sum())),
        ('แนะนำให้เก็บ', int(counts.get('KEEP', 0))), ('ควรตรวจสอบ', int(counts.get('REVIEW', 0))),
        ('แนะนำให้ลบ', int(counts.get('RECOMMEND_DELETE', 0))),
        ('พื้นที่ที่อาจประหยัด (MiB)', round(float(saved), 2)),
    ], columns=['รายการ', 'ผลลัพธ์'])

def show_photo_results(results, max_images=12, group_id=None):
    selected = results if group_id is None else results[results['similarityGroup'].eq(group_id)]
    selected = selected.assign(_group=selected['similarityGroup'].fillna('รูปเดี่ยว')).sort_values(['_group', 'qualityScore'], ascending=[True, False], kind='stable')
    if selected.empty:
        print('ไม่มีรูปตามเงื่อนไขที่เลือก')
        return
    limit = len(selected) if max_images is None else max(0, int(max_images))
    print(f'แสดง {min(limit, len(selected))} จาก {len(selected)} รูป (ผลคำนวณครบทุกภาพ)')
    previous = None
    for _, row in selected.head(limit).iterrows():
        group = str(row['_group'])
        if group != previous:
            print('\n' + '=' * 64)
            if pd.isna(row['similarityGroup']):
                basis = 'ไม่ได้เข้ากลุ่ม'
            elif pd.isna(row['taken_at']):
                basis = 'กลุ่มรูปไม่มีเวลา — ความคล้ายของภาพ'
            elif bool(row.get('timedPhashMatch', False)):
                basis = 'กลุ่มรูปมีเวลา — รวมเพิ่มจากความคล้ายเมื่อเวลาเกินกรอบ'
            else:
                basis = 'กลุ่มรูปที่ถ่ายใกล้กัน'
            print(f'{group} | {basis}')
            previous = group
        preview = Image.fromarray(load_image(Path(row['path']))['rgb'])
        preview.thumbnail((360, 270))
        display(preview)
        badge = ' | BEST SHOT (รอตรวจสอบ)' if bool(row['bestShot']) and bool(row['bestShotNeedsReview']) else ' | BEST SHOT' if bool(row['bestShot']) else ''
        print(f'{row.get("relative_path", row["filename"])}{badge}')
        print(f'คุณภาพ {row["qualityScore"]:.1f}/100 | ความคมชัด {row["sharpnessScore"]:.1f}/100 | แสง: {EXPOSURE_TEXT[row["exposure_label_pred"]]}')
        print(f'{RECOMMENDATION_TEXT[row["recommendation"]]} — {short_reason(row, results)}\n')

def export_results(results, pairs, params, output_root, label='demo'):
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    destination = Path(output_root) / f'{label}-{run_id}'
    destination.mkdir(parents=True, exist_ok=False)
    export = results.copy()
    if 'reasons' in export:
        export['reasons'] = export['reasons'].map(lambda value: json.dumps(value, ensure_ascii=False))
    export.to_csv(destination / 'pipeline_results.csv', index=False, encoding='utf-8-sig')
    pairs.to_csv(destination / 'similarity_pairs.csv', index=False, encoding='utf-8-sig')
    (destination / 'parameters_used.json').write_text(json.dumps(params, ensure_ascii=False, indent=2), encoding='utf-8')
    print('บันทึกผลใหม่ที่:', destination)
    return destination


In [ ]:
SELECTED_FOLDER = choose_image_folder(IMAGE_FOLDER)
print('โฟลเดอร์ที่เลือก:', SELECTED_FOLDER)
demo_features, demo_results, demo_pairs, demo_issues = analyze_folder(SELECTED_FOLDER, PARAMS)
display(summary_table(demo_results))
if not demo_issues.empty:
    print('ไฟล์ที่ข้ามเพราะอ่านไม่ได้:')
    display(demo_issues[['filename', 'error']])
show_photo_results(demo_results, max_images=MAX_PREVIEW_IMAGES)


### 8.1 ตรวจตารางผล / ลองโฟลเดอร์อื่น

ตารางเก็บคะแนนเต็มก่อนปัดเศษเหมือนฝั่งประมวลผลของแอป แสดงเฉพาะบางคอลัมน์ให้อ่านง่าย
เปลี่ยน `IMAGE_FOLDER` แล้วรัน cell ส่วน 8 ซ้ำได้โดยไม่ต้องรัน definitions ใหม่
หากต้องการดูเฉพาะกลุ่ม เช่น `show_photo_results(demo_results, group_id='G001')`


In [ ]:
result_columns = ['relative_path', 'taken_at', 'similarityGroup', 'timedPhashMatch', 'sharpnessScore', 'exposure_label_pred', 'qualityScore', 'bestShot', 'bestShotNeedsReview', 'recommendation']
display(demo_results[result_columns].round({'sharpnessScore': 2, 'qualityScore': 2}))

# เปิดเองหากต้องการดูหลักฐานเป็นรายคู่ ไม่ใช่ผลกลุ่มสุดท้าย
SHOW_PAIR_DETAILS = False
if SHOW_PAIR_DETAILS:
    display(demo_pairs)

if EXPORT_DEMO_RESULTS:
    demo_export_dir = export_results(demo_results, demo_pairs, PARAMS, OUTPUT_DIR)


## 9. ประเมินกับ Ground Truth (ทางเลือก)

ส่วน demo ก่อนหน้านี้ไม่ต้องมี `labels.csv` ส่วนนี้เปิดเมื่อพร้อมประเมิน dataset จริง
ใช้ชื่อคอลัมน์เดิม: `filename, source, scene_id, group_id, split`
และคอลัมน์สำหรับแต่ละโมดูล `blur_label, exposure_label, best_shot` ถ้ามี


**ประเภทข้อมูลและ class ที่ใช้ประเมิน**

| โมดูล | ประเภทภาพใน dataset ทั้งชุด | Class ที่ระบบทำนาย / การประเมิน |
|---|---|---|
| Sharpness | Sharp, Slight Blur, Motion Blur, Defocus Blur อย่างละ 30 ภาพ (รวม 120) | Sharp / Blur: Matrix 2×2 และผลตรวจจับแยกตาม 4 ประเภท |
| Exposure | Normal, Underexposed, Overexposed อย่างละ 30 ภาพ (รวม 90) | 3 class: Matrix 3×3 |
| Similarity | 20 กลุ่ม กลุ่มละ 5 ภาพ (รวม 100) | Group ID ไม่ใช่ class; ประเมินคู่ภาพ Same / Different group และ ARI |

จำนวนข้างต้นเป็นทั้ง dataset ส่วนผลประเมินใช้เฉพาะ `EVALUATION_SPLIT` ที่เลือก
Sharpness แปลง Slight/Motion/Defocus Blur เป็น `blur` เพื่อประเมิน 2 class โดยยังเก็บชนิดเดิมไว้แจกแจงผล
กราฟแสดงทั้งจำนวนจริงและเปอร์เซ็นต์ในแต่ละแถว: แถว = Ground Truth, คอลัมน์ = Prediction

- อ่าน split จาก CSV ไม่สุ่มแบ่งใหม่ และไม่แอบใช้ Final Test ปรับค่า
- Sharpness / Exposure: Confusion Matrix, Accuracy, Macro P/R/F1 และ Precision/Recall/F1 พร้อม support ราย class
- Grouping: ประเมิน **กลุ่มสุดท้าย** บน source similarity + unique เป็น pairwise P/R/F1
  พร้อม False Merge / False Split, Matrix แบบคู่ภาพ 2×2 และ Adjusted Rand Index (ARI); ไม่ใช้ `is_similar` ใน pair table แทนกลุ่มสุดท้าย
- Best Shot: ตรวจเฉพาะกลุ่ม GT ที่มีภาพที่ดีที่สุดเพียงหนึ่งรูป และนับกรณีเลือกผิด/ไม่เลือกด้วย
- จำนวน KEEP/REVIEW/DELETE เป็นผลการทำงาน ไม่ใช่ความถูกต้องของคำแนะนำลบ
- โหลด features จากภาพใหม่ ไม่ใช้ cache ของ notebook เก่าที่อาจอ่าน EXIF คนละแบบ
- ถ้าภาพ GT ไม่ครบหรืออ่านไม่ได้ จะหยุดการประเมินแทนการรายงานคะแนนจากชุดที่หายไปบางส่วน
- Final Test เดิมเคยถูกตรวจระหว่างพัฒนาแล้ว จึงรายงานเป็น retrospective diagnostics
  ไม่อ้างว่าเป็น untouched holdout; ตัวเลข F1 เก่าใน JSON ไม่ใช่ผลของ flow นี้


In [ ]:
VALID_DATA_SPLITS = {'development', 'final_test'}
REQUIRED_LABEL_COLUMNS = {'filename', 'source', 'scene_id', 'group_id', 'split'}


def canonical_name(filename):
    """Normalize harmless filename variations for matching image paths to labels."""
    if pd.isna(filename):
        return ""

    name = Path(str(filename).strip()).name.casefold()
    name = name.replace("sligth_blur", "slight_blur")
    name = name.replace("_ ", "_")

    if name.endswith(".jpeg"):
        name = name[:-5] + ".jpg"

    return name


def data_source(path):
    path = Path(path)

    if path.name.casefold().startswith("unique_"):
        return "unique"

    parts = [part.casefold() for part in Path(path).parts]
    aliases = {"natural_edge_cases": "natural", "natural": "natural"}
    if "dataset" in parts:
        index = parts.index("dataset")
        if index + 1 < len(parts):
            folder = parts[index + 1]
            if folder != "metadata":
                return aliases.get(folder, folder)
    for part in parts:
        if part in {"sharpness", "exposure", "similarity", "natural_edge_cases", "natural", "unique"}:
            return aliases.get(part, part)
    return "other"


def load_labels(verbose=True):
    if not LABELS_PATH.is_file():
        raise FileNotFoundError(f"Missing Ground Truth: {LABELS_PATH}")

    df = pd.read_csv(LABELS_PATH).dropna(how="all").copy()
    missing = REQUIRED_LABEL_COLUMNS.difference(df.columns)

    if missing:
        raise ValueError(
            f"labels.csv lacks columns: {sorted(missing)}. "
            "ใช้ Labels เวอร์ชันใหม่ที่มีคอลัมน์ split"
        )

    df["filename"] = df["filename"].astype("string").str.strip()
    df["source"] = (
        df["source"]
        .astype("string")
        .str.strip()
        .str.casefold()
        .replace({"natural_edge_cases": "natural"})
    )

    for col in ("scene_id", "group_id"):
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

    df["split"] = (
        df["split"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    unknown = df.loc[~df["split"].isin(VALID_DATA_SPLITS), "split"]
    if not unknown.empty:
        raise ValueError(
            f"Unknown/missing split values: {unknown.unique().tolist()}. "
            f"Expected {sorted(VALID_DATA_SPLITS)}"
        )

    df["file_key"] = df["filename"].map(canonical_name)
    df = df.loc[df["file_key"].ne("") & df["source"].notna()].copy()

    duplicates = df.duplicated(["source", "file_key"], keep=False)
    if duplicates.any():
        examples = df.loc[
            duplicates,
            ["source", "filename"]
        ].head(8).to_dict("records")
        raise ValueError(f"Duplicate label keys: {examples}")

    scene_overlap = (
        df.dropna(subset=["scene_id"])
        .groupby(["source", "scene_id"])["split"]
        .nunique()
    )
    if (scene_overlap > 1).any():
        raise ValueError(
            "A scene appears in both Development and Final Test: "
            f"{scene_overlap[scene_overlap > 1].index.tolist()[:8]}"
        )

    sim = df[df["source"].eq("similarity")]
    if sim["group_id"].isna().any():
        raise ValueError("Similarity contains missing group_id values")

    sim_overlap = sim.groupby("group_id")["split"].nunique()
    if (sim_overlap > 1).any():
        raise ValueError(
            "Similarity groups cannot cross Development / Final Test: "
            f"{sim_overlap[sim_overlap > 1].index.tolist()}"
        )

    if verbose:
        print(f"Validated labels: {len(df)} images")
        display(pd.crosstab(df["source"], df["split"]))
    return df


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, adjusted_rand_score

def prepare_labeled_features(split='development'):
    if split not in {'development', 'final_test', 'all'}:
        raise ValueError('split ต้องเป็น development, final_test หรือ all')
    labels = load_labels(verbose=False)
    selected = labels.copy() if split == 'all' else labels[labels['split'].eq(split)].copy()
    if selected.empty:
        raise ValueError(f'ไม่มี labels สำหรับ split={split}')
    file_index = {}
    for path in list_image_files(DATASET_ROOT):
        key = (data_source(path), canonical_name(path.name))
        if key in file_index:
            raise ValueError(f'ชื่อภาพซ้ำภายใน source เดียวกัน: {key}; แก้ความกำกวมก่อนประเมิน')
        file_index[key] = path
    missing = [(row.source, row.filename) for row in selected.itertuples() if (row.source, row.file_key) not in file_index]
    if missing:
        raise FileNotFoundError(f'ภาพ GT หาย {len(missing)} ภาพ ตัวอย่าง: {missing[:5]}')
    paths = [file_index[(row.source, row.file_key)] for row in selected.itertuples()]
    features, issues = extract_features_batch(paths, relative_to=DATASET_ROOT, show_progress=True)
    if not issues.empty:
        raise ValueError(f'ประเมินไม่ครบ เพราะมีภาพอ่านไม่ได้: {issues["filename"].tolist()}')
    features['source'] = features['path'].map(data_source)
    features['file_key'] = features['filename'].map(canonical_name)
    columns = ['source', 'file_key', 'scene_id', 'group_id', 'split'] + [key for key in ('blur_label', 'exposure_label', 'best_shot') if key in selected]
    joined = features.merge(selected[columns], on=['source', 'file_key'], how='left', validate='one_to_one')
    if len(joined) != len(selected) or joined['split'].isna().any():
        raise ValueError('ภาพกับ GT ไม่ตรงกันครบทุกแถว')
    return joined.reset_index(drop=True)

def classification_metrics(truth, prediction, labels, name):
    p, r, f1, _ = precision_recall_fscore_support(truth, prediction, labels=labels, average='macro', zero_division=0)
    return {'module': name, 'images': len(truth), 'accuracy': float(accuracy_score(truth, prediction)), 'macro_precision': float(p), 'macro_recall': float(r), 'macro_f1': float(f1)}

SHARPNESS_DATASET_TYPES = ['sharp', 'slight_blur', 'motion_blur', 'defocus_blur']
SHARPNESS_BINARY_MAP = {
    'sharp': 'sharp', 'slight_blur': 'blur', 'motion_blur': 'blur',
    'defocus_blur': 'blur', 'blur': 'blur',
}
EXPOSURE_CLASSES = ['normal', 'underexposed', 'overexposed']

def normalize_evaluation_labels(values, allowed, context):
    normalized = values.map(
        lambda value: '' if pd.isna(value) else
        '_'.join(str(value).strip().casefold().replace('-', ' ').replace('_', ' ').split())
    ).replace({'sligth_blur': 'slight_blur'})
    invalid = ~normalized.isin(allowed)
    if invalid.any():
        examples = values.loc[invalid].astype(str).unique().tolist()[:8]
        raise ValueError(f'{context}: label หายหรือไม่รองรับ {examples}; ตรวจ CSV ก่อนประเมิน')
    return normalized

def classification_outputs(truth, prediction, labels, display_labels, name):
    truth = normalize_evaluation_labels(truth, labels, f'{name} Ground Truth')
    prediction = normalize_evaluation_labels(prediction, labels, f'{name} predictions')
    if len(truth) != len(prediction):
        raise ValueError(f'{name}: จำนวน Ground Truth กับ predictions ไม่เท่ากัน')
    details = classification_report(
        truth, prediction, labels=labels, output_dict=True, zero_division=0
    )
    per_class = pd.DataFrame([
        {'module': name, 'class': shown, 'precision': details[label]['precision'],
         'recall': details[label]['recall'], 'f1': details[label]['f1-score'],
         'support': int(details[label]['support'])}
        for label, shown in zip(labels, display_labels)
    ])
    counts = confusion_matrix(truth, prediction, labels=labels)
    if int(counts.sum()) != len(truth):
        raise ValueError(f'{name}: มีภาพไม่ถูกนับใน Confusion Matrix')
    return classification_metrics(truth, prediction, labels, name), per_class, {
        'counts': counts, 'labels': display_labels, 'unit': 'images'
    }

def grouping_pair_labels(predicted):
    truth, assigned = [], []
    for i, j in combinations(range(len(predicted)), 2):
        a, b = predicted.iloc[i], predicted.iloc[j]
        truth.append(bool(
            a['source'] == b['source'] == 'similarity'
            and pd.notna(a['group_id']) and pd.notna(b['group_id'])
            and str(a['group_id']) == str(b['group_id'])
        ))
        assigned.append(bool(
            pd.notna(a['similarityGroup']) and pd.notna(b['similarityGroup'])
            and a['similarityGroup'] == b['similarityGroup']
        ))
    return np.asarray(truth, dtype=bool), np.asarray(assigned, dtype=bool)

def grouping_metrics(predicted):
    truth, assigned = grouping_pair_labels(predicted)
    if not len(truth):
        raise ValueError('Grouping evaluation ต้องมีอย่างน้อย 2 ภาพ')
    p, r, f1, _ = precision_recall_fscore_support(
        truth, assigned, average='binary', zero_division=0
    )
    broken_groups = 0
    for _, members in predicted[predicted['source'].eq('similarity')].groupby('group_id'):
        identifiers = [
            f'cluster:{row["similarityGroup"]}' if pd.notna(row['similarityGroup'])
            else f'singleton:{index}' for index, row in members.iterrows()
        ]
        broken_groups += len(set(identifiers)) > 1

    # An ungrouped image is its own singleton, never one shared "missing" cluster.
    truth_clusters, predicted_clusters = [], []
    for position, (_, row) in enumerate(predicted.iterrows()):
        truth_clusters.append(
            f'group:{row["group_id"]}'
            if row['source'] == 'similarity' and pd.notna(row['group_id'])
            else f'singleton:{position}'
        )
        predicted_clusters.append(
            f'cluster:{row["similarityGroup"]}' if pd.notna(row['similarityGroup'])
            else f'singleton:{position}'
        )
    return {
        'images': len(predicted), 'pairs': len(truth),
        'positive_pairs': int(truth.sum()), 'precision': float(p),
        'recall': float(r), 'f1': float(f1),
        'false_merge_pairs': int((~truth & assigned).sum()),
        'false_split_pairs': int((truth & ~assigned).sum()),
        'false_split_groups': int(broken_groups),
        'adjusted_rand_index': float(adjusted_rand_score(truth_clusters, predicted_clusters)),
    }

def best_shot_metrics(predicted):
    if 'best_shot' not in predicted:
        return {'evaluated_groups': 0, 'group_accuracy': None, 'note': 'ไม่มี best_shot Ground Truth'}
    correct, total = 0, 0
    for _, members in predicted[predicted['source'].eq('similarity')].groupby('group_id'):
        if len(members) < 2 or members['best_shot'].isna().any():
            continue
        truth = members['best_shot'].astype(str).str.strip().str.casefold().isin({'1', '1.0', 'true', 'yes'})
        if truth.sum() != 1:
            continue
        total += 1
        actual = members['bestShot'].astype(bool)
        correct += actual.sum() == 1 and actual.equals(truth)
    return {'evaluated_groups': total, 'group_accuracy': float(correct / total) if total else None}

def evaluate_current_pipeline(features, params):
    scored = add_quality_scores(features, params)
    reports, per_class_reports, distribution, subtype_rows = [], [], [], []
    matrices = {}

    sharp = scored[scored['source'].eq('sharpness')].copy()
    if not sharp.empty:
        if 'blur_label' not in sharp:
            raise ValueError('Sharpness evaluation ต้องมี blur_label ใน CSV')
        sharp['dataset_type'] = normalize_evaluation_labels(
            sharp['blur_label'], SHARPNESS_BINARY_MAP, 'Sharpness dataset types'
        )
        sharp['binary_truth'] = sharp['dataset_type'].map(SHARPNESS_BINARY_MAP)
        sharp['sharpness_label'] = normalize_evaluation_labels(
            sharp['sharpness_label'], ['sharp', 'blur'], 'Sharpness predictions'
        )
        overall, details, matrix = classification_outputs(
            sharp['binary_truth'], sharp['sharpness_label'],
            ['sharp', 'blur'], ['Sharp', 'Blur'], 'Sharpness (binary)'
        )
        reports.append(overall)
        per_class_reports.append(details)
        matrices['Sharpness (binary)'] = matrix
        types = SHARPNESS_DATASET_TYPES + (['blur'] if sharp['dataset_type'].eq('blur').any() else [])
        for kind in types:
            rows = sharp[sharp['dataset_type'].eq(kind)]
            expected = SHARPNESS_BINARY_MAP[kind]
            correct = int(rows['sharpness_label'].eq(expected).sum())
            distribution.append({
                'module': 'Sharpness', 'dataset_class': kind,
                'evaluation_class': expected, 'images': len(rows)
            })
            subtype_rows.append({
                'dataset_type': kind, 'expected_binary_class': expected,
                'images': len(rows), 'predicted_sharp': int(rows['sharpness_label'].eq('sharp').sum()),
                'predicted_blur': int(rows['sharpness_label'].eq('blur').sum()),
                'correct_binary_predictions': correct,
                'correct_rate': correct / len(rows) if len(rows) else np.nan,
            })

    exposure = scored[scored['source'].eq('exposure')].copy()
    if not exposure.empty:
        if 'exposure_label' not in exposure:
            raise ValueError('Exposure evaluation ต้องมี exposure_label ใน CSV')
        exposure['exposure_truth'] = normalize_evaluation_labels(
            exposure['exposure_label'], EXPOSURE_CLASSES, 'Exposure Ground Truth'
        )
        overall, details, matrix = classification_outputs(
            exposure['exposure_truth'], exposure['exposure_label_pred'], EXPOSURE_CLASSES,
            ['Normal', 'Underexposed', 'Overexposed'], 'Exposure'
        )
        reports.append(overall)
        per_class_reports.append(details)
        matrices['Exposure'] = matrix
        for label in EXPOSURE_CLASSES:
            distribution.append({
                'module': 'Exposure', 'dataset_class': label,
                'evaluation_class': label,
                'images': int(exposure['exposure_truth'].eq(label).sum()),
            })

    grouping_input = features[features['source'].isin(['similarity', 'unique'])].reset_index(drop=True)
    stages = []
    best = {'evaluated_groups': 0, 'group_accuracy': None}
    grouping_dataset = {
        'images': len(grouping_input),
        'ground_truth_groups': int(grouping_input.loc[grouping_input['source'].eq('similarity'), 'group_id'].nunique()),
        'unique_images': int(grouping_input['source'].eq('unique').sum()),
    }
    if len(grouping_input) >= 2:
        baseline = deepcopy(params)
        baseline['similarity']['timed_phash_threshold'] = None
        final_grouped = None
        for name, settings in [('Time + untimed pHash', baseline), ('App current (+ timed pHash)', params)]:
            predicted, _ = run_core_pipeline(grouping_input, settings)
            stages.append({'stage': name, **grouping_metrics(predicted)})
            final_grouped = predicted
        best = best_shot_metrics(final_grouped)
        pair_truth, pair_predicted = grouping_pair_labels(final_grouped)
        matrices['Similarity (pair relation)'] = {
            'counts': confusion_matrix(pair_truth, pair_predicted, labels=[False, True]),
            'labels': ['Different group', 'Same group'], 'unit': 'pairs',
        }

    full_results, _ = run_core_pipeline(features, params)
    return {
        'classification': pd.DataFrame(reports),
        'per_class': pd.concat(per_class_reports, ignore_index=True) if per_class_reports else pd.DataFrame(),
        'class_distribution': pd.DataFrame(distribution),
        'sharpness_by_type': pd.DataFrame(subtype_rows),
        'confusion_matrices': matrices,
        'grouping_dataset': grouping_dataset,
        'grouping_stages': pd.DataFrame(stages),
        'best_shot': best, 'full_results': full_results,
    }

def show_confusion_matrices(report):
    for name, result in report['confusion_matrices'].items():
        counts = np.asarray(result['counts'])
        totals = counts.sum(axis=1, keepdims=True)
        percentages = np.divide(
            counts * 100.0, totals, out=np.zeros_like(counts, dtype=float), where=totals != 0
        )
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        for ax, values, title, fmt in [
            (axes[0], counts, f'Count ({result["unit"]})', 'd'),
            (axes[1], percentages, 'Percent within each true class', '.1f'),
        ]:
            ConfusionMatrixDisplay(values, display_labels=result['labels']).plot(
                ax=ax, cmap='Blues', colorbar=False, values_format=fmt, xticks_rotation=20
            )
            ax.set_title(title)
            ax.set_xlabel('Predicted class' if result['unit'] == 'images' else 'Predicted pair relation')
            ax.set_ylabel('True class' if result['unit'] == 'images' else 'True pair relation')
        fig.suptitle(name, fontsize=14, weight='bold')
        fig.tight_layout()
        plt.show()
        plt.close(fig)

def show_evaluation_report(report):
    print('Class และจำนวนภาพใน split ที่เลือก (ไม่ใช่จำนวนทั้ง dataset):')
    display(report['class_distribution'])
    print('Classification summary — ค่า 0–1:')
    display(report['classification'].round(4))
    print('Precision / Recall / F1 ราย class และ support (จำนวนภาพจริง):')
    display(report['per_class'].round(4))
    print('Sharpness: แยกตาม 4 ประเภทภาพ แต่ทำนาย Sharp / Blur เพียง 2 class')
    print('correct_rate คือสัดส่วนทำนายคม–เบลอถูกภายในแต่ละประเภท ไม่ใช่การทำนายชนิดเบลอ')
    display(report['sharpness_by_type'].round(4))
    print('Confusion Matrix: แถว = คำตอบจริง, คอลัมน์ = ผลทำนาย; แถวที่ไม่มีภาพจะแสดง 0%')
    show_confusion_matrices(report)
    print('Similarity: Group ID ใช้ระบุกลุ่ม ไม่ใช่ class ของภาพ')
    print('ข้อมูลกลุ่มใน split นี้:', report['grouping_dataset'])
    print('Pairwise P/R/F1, False Merge / False Split และ Adjusted Rand Index (ARI):')
    display(report['grouping_stages'].round(4))
    print('ARI = 1 หมายถึงจัดกลุ่มตรงกัน แม้หมายเลขกลุ่มต่างกัน; ค่าอาจติดลบได้')
    print('Best Shot:', report['best_shot'])
    print('คำแนะนำของภาพทั้งหมด — เป็นจำนวนผล ไม่ใช่ accuracy ของการลบ:')
    display(summary_table(report['full_results']))


In [ ]:
evaluation_features = None
evaluation_report = None
if RUN_GROUND_TRUTH_EVALUATION:
    print(f'กำลังอ่านภาพและ Ground Truth: split={EVALUATION_SPLIT}', flush=True)
    evaluation_features = prepare_labeled_features(EVALUATION_SPLIT)
    print(f'ประเมินด้วย flow ปัจจุบัน: split={EVALUATION_SPLIT}, ภาพ {len(evaluation_features)} ภาพ')
    evaluation_report = evaluate_current_pipeline(evaluation_features, PARAMS)
    show_evaluation_report(evaluation_report)
else:
    print('ข้าม Ground Truth evaluation — ส่วนทดลองโฟลเดอร์ก่อนหน้านี้ใช้งานได้ตามปกติ')
    print('ต้องการประเมิน: ตั้ง RUN_GROUND_TRUTH_EVALUATION=True ในส่วน 2 แล้วรันส่วน 9')


## 10. ทดลอง parameter แบบแก้ตัวแปรแล้วดูตาราง (ไม่มี UI)

แก้ `TEST_PARAMS` ได้โดยไม่เปลี่ยน JSON ต้นฉบับ แล้วรันกับโฟลเดอร์เดิมเพื่อดูรูป/ผลใหม่
การเปรียบเทียบ metrics ทำได้เฉพาะ Development และใช้ features ชุดเดียวกัน
หากต้องการบันทึกให้เปิด `SAVE_TEST_RESULTS=True` จะสร้างโฟลเดอร์ผลใหม่พร้อม parameters_used.json
ไม่เขียนทับ parameter ของแอปหรือผลเก่า


In [ ]:
TEST_PARAMS = deepcopy(PARAMS)
# ตัวอย่างจุดปรับ (เอา # ออกเฉพาะบรรทัดที่ต้องการทดลอง):
# TEST_PARAMS['similarity']['timed_phash_threshold'] = 25
# TEST_PARAMS['similarity']['phash_threshold'] = 18
# TEST_PARAMS['similarity']['exif_time_window_seconds'] = 20
# TEST_PARAMS['quality_score'].update(sharpness_weight=0.7, exposure_weight=0.3)
# TEST_PARAMS['recommendation']['duplicate_quality_gap'] = 7
# TEST_PARAMS['recommendation']['best_shot_tie_margin'] = 0.5

RUN_FOLDER_PARAMETER_TEST = False
SAVE_TEST_RESULTS = False
if RUN_FOLDER_PARAMETER_TEST:
    validate_parameters(TEST_PARAMS)
    test_results, test_pairs = run_core_pipeline(demo_features, TEST_PARAMS)
    display(summary_table(test_results))
    show_photo_results(test_results, max_images=MAX_PREVIEW_IMAGES)
    if SAVE_TEST_RESULTS:
        export_results(test_results, test_pairs, TEST_PARAMS, OUTPUT_DIR, label='parameter-test')

if RUN_PARAMETER_COMPARISON:
    if evaluation_features is None or EVALUATION_SPLIT != 'development':
        raise ValueError('เปรียบเทียบ parameter บน Development เท่านั้น: เปิดการประเมินส่วน 9 และเลือก development ก่อน')
    comparison = []
    for name, settings in [('App current', PARAMS), ('Test parameters', TEST_PARAMS)]:
        report = evaluate_current_pipeline(evaluation_features, settings)
        for row in report['classification'].to_dict('records'):
            comparison.append({'candidate': name, **row})
        grouping = report['grouping_stages']
        if not grouping.empty:
            last = grouping.iloc[-1]
            comparison.append({'candidate': name, 'module': 'Grouping', 'images': last['images'], 'precision': last['precision'], 'recall': last['recall'], 'f1': last['f1'], 'false_merge_pairs': last['false_merge_pairs'], 'false_split_pairs': last['false_split_pairs']})
    display(pd.DataFrame(comparison).round(4))


## 11. จุดเล่าสำหรับ present และบทบาทของแอป

| ลำดับ | สิ่งที่อธิบาย / ทดลอง |
|---|---|
| 1 | ปัญหารูปซ้ำ/รูปคล้าย และเป้าหมายการให้คำแนะนำ |
| 2 | ภาพรวม flow และค่าจาก best_parameters.json |
| 3 | โหลดภาพ, Sharpness, Exposure, Quality Score |
| 4 | แยกรูปมีเวลา/ไม่มีเวลา และเหตุผลที่มี pHash สอง threshold |
| 5 | คงกลุ่มเวลาเดิม แล้วตรวจ timed pHash สำหรับรูปเกินกรอบ |
| 6 | Best Shot, คะแนนสูสี และเงื่อนไขแนะนำลบ |
| 7 | เลือกโฟลเดอร์จริง แล้วดูรูป + ข้อความสั้น ๆ ในส่วน 8 |
| 8 | ถ้ามี GT: แสดงผลที่รันใหม่จากส่วน 9 และลองค่าในส่วน 10 |
| 9 | เปิดเว็บ Smart Gallery: Next.js บน Vercel ส่งภาพให้ FastAPI บน Render แล้วแสดงผลเดียวกัน |

**บทบาทแยกกัน:** Notebook ใช้เปิดกระบวนการและทดลองให้เห็นผลก่อน;
แอปเพิ่มการเลือกรูป/โฟลเดอร์, เก็บ session ใน browser และหน้าจอสำหรับใช้งานจริง
โดยยังใช้กระบวนการวิเคราะห์เดียวกัน

**ข้อจำกัดที่ควรอธิบาย:** เวลาใกล้กันไม่ได้รับประกันว่าภาพเหมือนกัน, pHash อาจจับคู่ผิด
หรือพลาดเมื่อภาพเปลี่ยนมุมมาก, Quality Score ไม่ได้ตัดสินองค์ประกอบภาพ,
การเปรียบเทียบคู่ภาพมีต้นทุน O(n²), ระบบให้คำแนะนำและไม่ลบต้นฉบับ

**ประวัติผลเดิม:** Notebook ต้นฉบับทดลอง Strict EXIF + pHash / Average-link /
Singleton Rescue / Group Merge ด้วยพารามิเตอร์และ flow คนละชุด ตัวเลข Development
F1 ที่เก็บใน JSON จึงเป็นหลักฐานการทดลองเดิม ไม่ใช่ผลประเมินใหม่ของ Notebook นี้
ผลจริงของ flow ปัจจุบันจะเกิดเมื่อรันส่วน 9 กับ dataset/labels ที่ครบเท่านั้น
